# Phase 4 – Court Judgment Prediction (CJPE) with the Phase-1 legal LM
Binary classification of long Indian court judgments (0/1). Train on `train.json`, select on `dev.json`, report on `test.json`.

**Pipeline:** data audit -> tokenise once (cache) -> representation ablation (LoRA probe, subset) -> fine-tuning-method LR grid (subset) -> full-data runs of every method (best dev checkpoint kept) -> layer-freezing ablation -> knowledge distillation -> dev/test evaluation -> submission.

# Citations
1) LoRMA: Low-Rank Multiplicative Adaptation for LLMs, Bihany et. al.
2) LoRA: Low-Rank Adaptation of Large Language Models, Hu et. al.
3) AdaLoRA: Adaptive Budget Allocation for Parameter-Efficient Fine-Tuning, Zhang et. al.
4) Layer freezing / unfreezing strategies (INT, BOT, TOP, ADT-L, ADT-H)
5) Distilling the Knowledge in a Neural Network, Hinton et. al.

In [ ]:
!find /kaggle/input -name final_model.pt

In [ ]:
!pip install ijson
import os
import sys
import json
import math
import time
import random
import zipfile
from typing import List, Tuple, Dict, Any, Generator
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import numpy as np, ijson
from multiprocessing import Pool
import regex
!pip install -q peft scikit-learn pandas
!pip uninstall -y torchao

In [ ]:
HF_TOKEN = None
if "HF_TOKEN" in os.environ:
    HF_TOKEN = os.environ["HF_TOKEN"]
else:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        HF_TOKEN = user_secrets.get_secret("HF")
        os.environ["HF_TOKEN"] = HF_TOKEN
        print("Successfully loaded HF_TOKEN from Kaggle Secrets.")
    except Exception as e:
        print(f"Note: Kaggle secrets not detected or HF key missing ({e}). Will attempt public access.")

if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Huggingface hub authentication successful.")
    except Exception as err:
        print(f"HF Login warning: {err}")


print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
from huggingface_hub import HfApi

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

HF_TOKENIZER_UPLOAD_REPO = os.environ.get("HF_TOKENIZER_UPLOAD_REPO", "legal-bpe-tokenizer-phase1")



TRAIN_FILES = ["Phase_4_CJPE/Train/train.json", "Phase_4_CJPE/Train/dev.json"]
# "Phase_4_CJPE/Train/dev.json"

TEST_FILES = ["Phase_4_CJPE/Test/test.json",]

VOCAB_SIZE = 33000
SEED = 67
random.seed(SEED)
torch.manual_seed(SEED)


HF_TOK_REPO = f"{HfApi(token=HF_TOKEN).whoami()['name']}/legal-bpe-tokenizer-phase2-v1"

OUTPUT_DIR = "/kaggle/working/cs781_phase3" if os.path.exists("/kaggle") else "./outputs"
TOKENIZER_DIR = os.path.join(OUTPUT_DIR, "custom_bpe_tokenizer_v2")
TOKEN_DATA_DIR = os.path.join(OUTPUT_DIR, "token_data_v3")

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "model_checkpoint")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)
os.makedirs(TOKEN_DATA_DIR, exist_ok=True)


TRAIN_BIN = os.path.join(TOKEN_DATA_DIR, "train.bin")
VAL_BIN   = os.path.join(TOKEN_DATA_DIR, "val.bin")

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
from transformers import PreTrainedTokenizerFast
from huggingface_hub import hf_hub_download

def download_data():
    local_train_paths = []
    print("\nDownloading phase 2 training data.")
    for tf in TRAIN_FILES:
        try:
            p = hf_hub_download(repo_id = HF_DATASET_REPO, filename = tf, repo_type = "dataset", token = HF_TOKEN)
            local_train_paths.append(p)
            print(f"Downloaded: {tf}")
        except Exception as e:
            print(f"Warning downloading {tf}: {e}")
            
    test_paths = []
    for tf in TEST_FILES:
        try:
            p = hf_hub_download(repo_id = HF_DATASET_REPO, filename = tf, repo_type = "dataset", token = HF_TOKEN)
            test_paths.append(p); print(f"Downloaded Test File: {tf}")
        except Exception as e:
            print(f"Warning downloading {tf}: {e}")
    return local_train_paths, test_paths

In [ ]:
train_paths, test_paths = download_data()

# inspect_json_files(train_paths, n=2)

In [ ]:
P = {os.path.basename(p): p for p in train_paths}
TEST = test_paths
print(P.keys(), TEST)

In [ ]:
%%writefile bpe_tokenizer.py
#!/usr/bin/env python3
import os, json, regex
from typing import Dict, List, Tuple
from collections import Counter, defaultdict
from tqdm.auto import tqdm

import regex


_PAT = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")


class PureByteLevelBPETokenizer:
    def __init__(self, vocab_size: int = 33000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]

        self.encoder: Dict[bytes, int] = {}
        self.decoder: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], bytes] = {}       # pair -> merged token
        self.merge_ranks: Dict[Tuple[bytes, bytes], int] = {}    # pair -> order learned (lower = earlier)
        self._cache: Dict[str, List[int]] = {}

        self._init_base_vocab()

    def _init_base_vocab(self):
        for b in range(256):
            b_bytes = bytes([b])
            self.encoder[b_bytes] = b
            self.decoder[b] = b_bytes

        # special tokens
        for idx, token in enumerate(self.special_tokens, start=256):
            b_bytes = token.encode("utf-8")
            self.encoder[b_bytes] = idx
            self.decoder[idx] = b_bytes

    @property
    def pad_token_id(self) -> int:
        return self.encoder["<pad>".encode("utf-8")]

    @property
    def unk_token_id(self) -> int:
        return self.encoder["<unk>".encode("utf-8")]

    @property
    def bos_token_id(self) -> int:
        return self.encoder["<s>".encode("utf-8")]

    @property
    def eos_token_id(self) -> int:
        return self.encoder["</s>".encode("utf-8")]

    # ------------------------------------------------------------------ training
    @staticmethod
    def _merge_word(word, a, b, new):
        out, i, n = [], 0, len(word)
        while i < n:
            if i < n - 1 and word[i] == a and word[i + 1] == b:
                out.append(new)
                i += 2
            else:
                out.append(word[i])
                i += 1
        return out

    def train_from_texts(self, texts: List[str], target_vocab_size: int = None, min_word_freq: int = 2):
        if target_vocab_size is None:
            target_vocab_size = self.vocab_size
        num_merges = target_vocab_size - len(self.encoder)
        print(f"\nTraining byte level BPE from scratch. Target merges: {num_merges:,} | Base Vocab: {len(self.encoder)}")

        # 1) pre-tokenize + count unique words (the BPE then runs on this small table, not the raw text)
        word_counts = Counter()
        for text in tqdm(texts, desc="Counting pre-tokens"):
            if text and text.strip():
                word_counts.update(_PAT.findall(text))

        words, freqs = [], []
        for w, c in word_counts.items():
            if c >= min_word_freq:
                words.append([bytes([b]) for b in w.encode("utf-8", "replace")])
                freqs.append(c)
        del word_counts
        print(f"Unique words used for merge learning: {len(words):,}")

        # 2) initial pair statistics (updated incrementally afterwards)
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for i, w in enumerate(tqdm(words, desc="Initial pair counts")):
            for p in zip(w, w[1:]):
                pair_counts[p] += freqs[i]
                where[p].add(i)

        # 3) merge loop
        pbar = tqdm(total=num_merges, desc="Learning BPE Merges", dynamic_ncols=True)
        for _ in range(num_merges):
            if not pair_counts:
                print("No more candidate pairs found to merge.")
                break

            best_pair, freq = max(pair_counts.items(), key=lambda kv: kv[1])
            if freq < 2:
                print(f"Stopping early: Most frequent pair frequency is {freq}")
                break

            a, b = best_pair
            merged_token = a + b
            self.merges[best_pair] = merged_token
            self.merge_ranks[best_pair] = len(self.merge_ranks)
            if merged_token not in self.encoder:          # two merge paths can give the same bytes
                new_id = len(self.encoder)
                self.encoder[merged_token] = new_id
                self.decoder[new_id] = merged_token

            for i in list(where[best_pair]):
                w, f = words[i], freqs[i]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        del pair_counts[p]
                w = self._merge_word(w, a, b, merged_token)
                words[i] = w
                for p in zip(w, w[1:]):
                    pair_counts[p] += f
                    where[p].add(i)
            del where[best_pair]
            pair_counts.pop(best_pair, None)

            pbar.update(1)
            pbar.set_postfix({"vocab_size": len(self.encoder), "top_pair_freq": freq})

        pbar.close()
        self._cache.clear()
        print(f"Training finished! Final Vocab Size: {len(self.encoder):,}")

    # ------------------------------------------------------------------ encode / decode
    def _encode_word(self, word: str) -> List[int]:
        cached = self._cache.get(word)
        if cached is not None:
            return cached
        tokens = [bytes([b]) for b in word.encode("utf-8", "replace")]
        while len(tokens) >= 2:
            best_pair, best_rank = None, None
            for p in zip(tokens, tokens[1:]):
                r = self.merge_ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_pair, best_rank = p, r
            if best_pair is None:
                break
            tokens = self._merge_word(tokens, best_pair[0], best_pair[1], self.merges[best_pair])
        ids = [self.encoder.get(t, self.unk_token_id) for t in tokens]
        if len(self._cache) < 1_000_000:
            self._cache[word] = ids
        return ids

    def encode(self, text: str) -> List[int]:
        ids: List[int] = []
        for word in _PAT.findall(text):
            ids.extend(self._encode_word(word))
        return ids

    def decode(self, token_ids: List[int]) -> str:
        raw_bytes = b"".join(self.decoder.get(i, b"<unk>") for i in token_ids)
        return raw_bytes.decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ save / load
    def save(self, save_dir: str):
        os.makedirs(save_dir, exist_ok=True)
        vocab_path = os.path.join(save_dir, "vocab.json")
        merges_path = os.path.join(save_dir, "merges.json")
        config_path = os.path.join(save_dir, "tokenizer_config.json")

        # latin1 maps bytes 1:1 onto code points, so any byte string survives JSON
        ser_vocab = {k.decode("latin1"): v for k, v in self.encoder.items()}
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(ser_vocab, f, ensure_ascii=False, indent=2)

        # JSON (not a space-separated text file): tokens can contain spaces, newlines, \r, \xa0 ...
        ser_merges = [[p1.decode("latin1"), p2.decode("latin1")] for (p1, p2) in self.merges]  # in learned order
        with open(merges_path, "w", encoding="utf-8") as f:
            json.dump(ser_merges, f, ensure_ascii=False)

        config = {
            "tokenizer_class": "PureByteLevelBPETokenizer",
            "vocab_size": len(self.encoder),
            "special_tokens": self.special_tokens,
            "pad_token_id": self.pad_token_id,
            "bos_token_id": self.bos_token_id,
            "eos_token_id": self.eos_token_id
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

        print(f"Tokenizer saved successfully to {save_dir}")

    @classmethod
    def load(cls, save_dir: str) -> "PureByteLevelBPETokenizer":
        tok = cls()
        with open(os.path.join(save_dir, "vocab.json"), "r", encoding="utf-8") as f:
            ser_vocab = json.load(f)
        tok.encoder = {k.encode("latin1"): v for k, v in ser_vocab.items()}
        tok.decoder = {v: k.encode("latin1") for k, v in ser_vocab.items()}

        tok.merges, tok.merge_ranks = {}, {}
        with open(os.path.join(save_dir, "merges.json"), "r", encoding="utf-8") as f:
            for rank, (s1, s2) in enumerate(json.load(f)):
                p1, p2 = s1.encode("latin1"), s2.encode("latin1")
                tok.merges[(p1, p2)] = p1 + p2
                tok.merge_ranks[(p1, p2)] = rank
        return tok

In [ ]:
%%writefile llama_scratch.py
"""
Llama-style decoder-only LM, written from scratch (no HF / no llama library).

Pieces: RMSNorm, RoPE (or ALiBi), grouped-query attention (GQA; MQA = n_kv_heads 1,
MHA = n_kv_heads == n_heads), SwiGLU MLP, pre-norm residual blocks, tied embeddings,
KV-cache generation with left-padded batches.

Only torch tensors/autograd/nn.Linear/nn.Embedding are used as building blocks.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------- config
@dataclass
class Config:
    vocab_size: int = 33000
    d_model: int = 1024
    n_layers: int = 16
    n_heads: int = 16
    n_kv_heads: int = 4          # 16 query heads share 4 KV heads (4 queries per KV head)
    ffn_hidden: int = 2560
    max_seq_len: int = 1024      # training context
    rope_cache_len: int = 4096   # RoPE table length (>= prompt + generated tokens)
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    tie_embeddings: bool = True
    pos_enc: str = "rope"        # "rope" | "alibi"
    attn_impl: str = "manual"    # "manual" (hand-written softmax(QK^T)V) | "sdpa" (torch fused kernel, same math)


PRESETS = {
    # exact totals with vocab 33000, tied embeddings
    "target_150m": dict(d_model=896, n_layers=14, n_heads=14, n_kv_heads=2, ffn_hidden=2304), # ~149.8M 
    "base":  dict(d_model=1024, n_layers=16, n_heads=16, n_kv_heads=4, ffn_hidden=2560),  # ~201.6M
    "small": dict(d_model=1024, n_layers=12, n_heads=16, n_kv_heads=4, ffn_hidden=2816),  # ~169.1M
    "tiny":  dict(d_model=64,   n_layers=2,  n_heads=4,  n_kv_heads=2, ffn_hidden=128),   # smoke tests
}


def analytic_param_count(c: Config) -> int:
    """Closed-form parameter count (use it to size a model before building it)."""
    hd = c.d_model // c.n_heads
    attn = 2 * c.d_model * c.n_heads * hd + 2 * c.d_model * c.n_kv_heads * hd   # Wq,Wo + Wk,Wv
    mlp = 3 * c.d_model * c.ffn_hidden                                           # gate, up, down
    per_layer = attn + mlp + 2 * c.d_model                                       # + 2 RMSNorm
    total = per_layer * c.n_layers + c.vocab_size * c.d_model + c.d_model        # + embedding + final norm
    if not c.tie_embeddings:
        total += c.vocab_size * c.d_model
    return total


def count_params(model, verbose=True) -> int:
    """model.parameters() yields a tied (shared) tensor only once, so tied weights are counted once."""
    total = sum(p.numel() for p in model.parameters())
    if verbose:
        emb = model.tok_emb.weight.numel()
        blocks = sum(p.numel() for p in model.layers.parameters())
        print(f"total {total/1e6:.1f}M | transformer blocks {blocks/1e6:.1f}M | embedding {emb/1e6:.1f}M "
              f"({'tied with output head' if model.cfg.tie_embeddings else 'output head separate'}) "
              f"| analytic {analytic_param_count(model.cfg)/1e6:.1f}M")
    return total


# ----------------------------------------------------------------------------- RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()                                   # always normalise in fp32
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)


# ----------------------------------------------------------------------------- RoPE
def build_rope_cache(head_dim, max_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))   # (hd/2,)
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)                                                # (max_len, hd/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x: (B, H, T, hd); cos/sin: (T, hd/2). Rotates the two halves of each head vector (Llama layout)."""
    half = x.shape[-1] // 2
    xf = x.float()
    x1, x2 = xf[..., :half], xf[..., half:]
    cos = cos[None, None]
    sin = sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


# ----------------------------------------------------------------------------- ALiBi
def get_alibi_slopes(n_heads):
    def pow2(n):
        start = 2 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_heads).is_integer():
        s = pow2(n_heads)
    else:
        c = 2 ** math.floor(math.log2(n_heads))
        s = pow2(c) + pow2(2 * c)[0::2][: n_heads - c]
    return torch.tensor(s, dtype=torch.float32)


# ----------------------------------------------------------------------------- GQA
def repeat_kv(x, n_rep):
    """(B, n_kv, S, hd) -> (B, n_kv*n_rep, S, hd); query head h uses kv head h // n_rep."""
    if n_rep == 1:
        return x
    B, n_kv, S, hd = x.shape
    return x[:, :, None].expand(B, n_kv, n_rep, S, hd).reshape(B, n_kv * n_rep, S, hd)


class GQAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.nh, self.nkv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.n_rep = self.nh // self.nkv
        self.impl = cfg.attn_impl
        self.wq = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)    # (B, nh,  T, hd)
        k = self.wk(x).view(B, T, self.nkv, self.hd).transpose(1, 2)   # (B, nkv, T, hd)
        v = self.wv(x).view(B, T, self.nkv, self.hd).transpose(1, 2)

        if cos is not None:                                            # RoPE on q and k (not v)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past is not None:                                           # cache holds only nkv heads
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        new_cache = (k, v) if use_cache else None
        S = k.shape[2]

        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # relative distance of every (query, key) pair; keys j <= query position are visible
        qpos = torch.arange(S - T, S, device=x.device)
        kpos = torch.arange(S, device=x.device)
        rel = qpos[:, None] - kpos[None, :]                            # (T, S)
        allowed = (rel >= 0)[None, None]                               # (1, 1, T, S)
        if key_valid is not None:                                      # hide left-padding keys
            allowed = allowed & key_valid[:, None, None, :]
        bias = None
        if alibi_slopes is not None:
            bias = -alibi_slopes.view(1, -1, 1, 1) * rel.clamp(min=0).float()[None, None]   # (1, nh, T, S)

        if self.impl == "sdpa" and key_valid is None and T == S:      # training fast path
            if bias is None:
                out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            else:
                m = bias.masked_fill(~allowed, torch.finfo(q.dtype).min).to(q.dtype)
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        else:                                                          # hand-written attention
            scores = ((q * (self.hd ** -0.5)) @ k.transpose(-2, -1)).float()   # (B, nh, T, S)
            if bias is not None:
                scores = scores + bias
            scores = scores.masked_fill(~allowed, -1e30)               # finite, so a fully-masked pad row can't give NaN
            probs = torch.softmax(scores, dim=-1).to(v.dtype)
            out = probs @ v                                            # (B, nh, T, hd)

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.wo(out), new_cache


# ----------------------------------------------------------------------------- SwiGLU
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden, bias=False)
        self.w_up = nn.Linear(d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


# ----------------------------------------------------------------------------- block + model
class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg.d_model, cfg.ffn_hidden)

    def forward(self, x, cos, sin, alibi_slopes, key_valid, past, use_cache):
        h, new_cache = self.attn(self.attn_norm(x), cos, sin, alibi_slopes, key_valid, past, use_cache)
        x = x + h
        x = x + self.mlp(self.mlp_norm(x))
        return x, new_cache


class LlamaLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.pos_enc in ("rope", "alibi") and cfg.attn_impl in ("manual", "sdpa")
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        if cfg.pos_enc == "rope":
            cos, sin = build_rope_cache(cfg.d_model // cfg.n_heads, cfg.rope_cache_len, cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
            self.alibi_slopes = None
        else:
            self.rope_cos = self.rope_sin = None
            self.register_buffer("alibi_slopes", get_alibi_slopes(cfg.n_heads), persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():       # GPT-2 style: shrink the residual-branch output projections
            if name.endswith("wo.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False, cache=None, start_pos=0,
                key_valid=None, all_logits=False):
        """
        training:  model(x, targets=y) -> scalar loss
        inference: model(idx, use_cache=True, cache=..., start_pos=..., key_valid=...) -> (logits, cache)
                   logits are for the last position only unless all_logits=True.
        """
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos = sin = None
        if self.cfg.pos_enc == "rope":
            assert start_pos + T <= self.cfg.rope_cache_len, "sequence longer than the RoPE table"
            cos = self.rope_cos[start_pos:start_pos + T]
            sin = self.rope_sin[start_pos:start_pos + T]

        new_caches = []
        for i, layer in enumerate(self.layers):
            past = cache[i] if cache is not None else None
            x, c = layer(x, cos, sin, self.alibi_slopes, key_valid, past, use_cache)
            new_caches.append(c)
        x = self.norm(x)

        if targets is not None:
            logits = self.lm_head(x)
            return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        logits = self.lm_head(x if all_logits else x[:, -1:, :])
        return logits, (new_caches if use_cache else None)


# ----------------------------------------------------------------------------- generation
@torch.no_grad()
def generate(model, prompts, max_new_tokens, eos_id, pad_id, temperature=0.1, top_k=30,
             top_p=0.95, rep_penalty=1.2, ban_eos=True):
    """
    prompts: list of lists of token ids (any lengths). Left-pads them into one batch, runs one prefill,
    then decodes with a KV cache. Returns a list of lists of generated ids (EOS / padding removed).
    temperature <= 0 -> greedy.
    """
    model.eval()
    device = next(model.parameters()).device
    V = model.cfg.vocab_size
    B = len(prompts)
    L = max(len(p) for p in prompts)
    ids = torch.full((B, L), pad_id, dtype=torch.long, device=device)
    key_valid = torch.zeros(B, L, dtype=torch.bool, device=device)
    seen = torch.zeros(B, V, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        t = torch.tensor(p, dtype=torch.long, device=device)
        ids[b, L - len(p):] = t
        key_valid[b, L - len(p):] = True
        seen[b, t] = True

    out = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    amp = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda"))
    with amp:
        logits, cache = model(ids, use_cache=True, key_valid=key_valid, start_pos=0)
        for step in range(max_new_tokens):
            lg = logits[:, -1, :].float()
            if ban_eos:
                lg[:, eos_id] = float("-inf")
            if rep_penalty != 1.0:                                     # CTRL-style repetition penalty
                pen = torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty)
                lg = torch.where(seen, pen, lg)
            if temperature <= 0:
                nxt = lg.argmax(-1)
            else:
                lg = lg / temperature
                if top_k and top_k > 0:
                    kth = torch.topk(lg, min(top_k, V)).values[:, -1:]
                    lg = lg.masked_fill(lg < kth, float("-inf"))
                if top_p < 1.0:
                    sl, si = torch.sort(lg, descending=True, dim=-1)
                    pr = torch.softmax(sl, dim=-1)
                    drop = (pr.cumsum(-1) - pr) > top_p                # keep the token that crosses top_p
                    sl = sl.masked_fill(drop, float("-inf"))
                    lg = torch.full_like(lg, float("-inf")).scatter(1, si, sl)
                nxt = torch.multinomial(torch.softmax(lg, dim=-1), 1).squeeze(1)

            nxt = torch.where(finished, torch.full_like(nxt, pad_id), nxt)
            for b in range(B):
                if not finished[b] and nxt[b].item() != eos_id:
                    out[b].append(int(nxt[b].item()))
            finished = finished | (nxt == eos_id)
            seen.scatter_(1, nxt[:, None], True)
            if bool(finished.all()):
                break
            key_valid = torch.cat([key_valid, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)
            logits, cache = model(nxt[:, None], use_cache=True, cache=cache,
                                  start_pos=L + step, key_valid=key_valid)
    return out

In [ ]:
from huggingface_hub import hf_hub_download
for f in ["vocab.json","merges.json","tokenizer_config.json"]:
    hf_hub_download("DumbLad/legal-bpe-tokenizer-phase1", f, repo_type="model",
                    local_dir="/kaggle/working/tok_phase1", token=HF_TOKEN)

In [ ]:
import os
import json
import ijson
import numpy as np
from collections import Counter

# Use the paths your previous notebook cells already downloaded.
TRAIN_PATH = P["train.json"]
DEV_PATH   = P["dev.json"]
TEST_PATH  = TEST[0]

def audit_json_array(path, name, show_samples=3):
    label_counts = Counter()
    lengths = []
    ids = set()
    duplicate_ids = 0
    missing_id = 0
    missing_text = 0
    missing_label = 0
    empty_text = 0
    samples = []

    print(f"\n{'='*65}\n{name}: {os.path.basename(path)}")
    print(f"File size: {os.path.getsize(path) / (1024**2):.2f} MiB")

    with open(path, "rb") as f:
        for record in ijson.items(f, "item"):
            record_id = record.get("id")
            text = record.get("text")
            label = record.get("label")

            if record_id is None:
                missing_id += 1
            elif record_id in ids:
                duplicate_ids += 1
            else:
                ids.add(record_id)

            if text is None:
                missing_text += 1
                text = ""
            text = str(text)
            lengths.append(len(text))

            if not text.strip():
                empty_text += 1

            if label is None:
                missing_label += 1
            else:
                # Keep the original label value for now; inspect unexpected values.
                label_counts[str(label)] += 1

            if len(samples) < show_samples:
                samples.append({
                    "id": record_id,
                    "label": label,
                    "text_preview": text[:250]
                })

    lengths = np.asarray(lengths, dtype=np.int64)
    print("Records:", len(lengths))
    print("Label counts:", dict(label_counts))
    if len(lengths):
        print("Label proportions:",
              {k: round(v / len(lengths), 4)
               for k, v in label_counts.items()})
        print("Text length in characters:")
        for q in [0, 25, 50, 75, 90, 95, 99, 100]:
            print(f"  P{q:02}: {np.percentile(lengths, q):,.0f}")
    print("Missing IDs:", missing_id)
    print("Duplicate IDs:", duplicate_ids)
    print("Missing text fields:", missing_text)
    print("Empty/whitespace-only texts:", empty_text)
    print("Missing labels:", missing_label)
    print("Sample records:")
    for s in samples:
        print(json.dumps(s, ensure_ascii=False, indent=2))

    return {
        "ids": ids,
        "labels": label_counts,
        "count": len(lengths),
        "lengths": lengths
    }

train_audit = audit_json_array(TRAIN_PATH, "TRAIN")
dev_audit   = audit_json_array(DEV_PATH, "DEV / VALIDATION")
test_audit  = audit_json_array(TEST_PATH, "TEST")
print("\nTrain/dev ID overlap:",
      len(train_audit["ids"] & dev_audit["ids"]))
print("Train/test ID overlap:",
      len(train_audit["ids"] & test_audit["ids"]))
print("Dev/test ID overlap:",
      len(dev_audit["ids"] & test_audit["ids"]))

## Stage 0 – extra data checks (decide preprocessing / imbalance handling)

## Code files (written once, imported by every job)

In [ ]:
%%writefile lorma_hook.py
import math, torch, torch.nn as nn, torch.nn.functional as F

_IDX = {}
def _rot_idx(d, device):                      # idx[i, j] = (j - i) mod d  -> row i rotated right by i
    k = (d, str(device))
    if k not in _IDX:
        ar = torch.arange(d, device=device)
        _IDX[k] = (ar[None, :] - ar[:, None]) % d
    return _IDX[k]

class LoRMALinear(nn.Module):
    def __init__(self, lin: nn.Linear, r, variant, alpha):
        super().__init__()
        assert lin.bias is None                # your Llama layers are bias-free
        self.base, self.variant, self.r = lin, variant, r
        self.scale = alpha / r
        for p in self.base.parameters(): p.requires_grad = False
        d = lin.out_features
        self.A = nn.Parameter(torch.zeros(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))
        if variant == "plus":                  # LoRMA+: B = 0, A random
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        else:                                  # LoRMAπ: pi(BA) = I at init
            nn.init.normal_(self.B, std=0.02)
            with torch.no_grad():
                self.B[:, 0] = 1.0
                self.A.zero_(); self.A[0, 0] = 1.0

    def forward(self, x):
        y0 = self.base(x)                      # W0 x   (multiplication is applied to the output dim)
        if self.variant == "plus":
            return y0 + self.scale * F.linear(F.linear(y0, self.A), self.B)
        M = self.B @ self.A                    # d x d, rank <= r
        P = M.gather(1, _rot_idx(M.size(0), M.device))   # permutation-based rank inflation
        return F.linear(y0, P)                 # = P @ y0

def apply_lorma(base, r=16, target_modules=("wq","wk","wv","wo","w_gate","w_up","w_down"), variant="plus", alpha=None):
    for p in base.parameters(): p.requires_grad = False
    alpha = alpha or 2 * r
    for _, mod in list(base.named_modules()):
        for cname, child in list(mod.named_children()):
            if cname in target_modules and isinstance(child, nn.Linear):
                setattr(mod, cname, LoRMALinear(child, r, variant, alpha))
    return base

In [ ]:
%%writefile cjpe_data.py
import os, re, json, hashlib
import numpy as np
from multiprocessing import Pool

MAX_CHARS = 80000          # keep first 40k + last 40k chars of very long judgments (~1.5% of train docs)
DECISION_WORDS = ["dismissed", "allowed", "acquitted", "convicted", "upheld", "affirmed", "restored",
                  "quashed", "remanded", "guilty", "liable", "rejected", "granted", "partly", "decreed", "sustained"]


# ------------------------------------------------------------------ preprocessing
def clean(s, pre):
    """raw  : whitespace collapse only
       norm : + lowercase + 'No.123' -> 'number 123'  (makes mixed-case train docs look like the lowercased dev/test docs)
       mask : norm + dates -> <date>, numbers with >=3 digits -> <num>"""
    s = s.replace("\u00a0", " ")
    if len(s) > MAX_CHARS:
        s = s[: MAX_CHARS // 2] + " " + s[-MAX_CHARS // 2:]
    if pre != "raw":
        s = s.lower()
        s = re.sub(r"\bno\.\s*(?=\d)", "number ", s)
    if pre == "mask":
        s = re.sub(r"\b\d{1,2}[-./]\d{1,2}[-./]\d{2,4}\b", " <date> ", s)
        s = re.sub(r"\d{3,}", " <num> ", s)
    return re.sub(r"\s+", " ", s).strip()


# ------------------------------------------------------------------ tokenisation cache (done ONCE per split x preprocessing)
_TOK = None
def _init(tok_dir):
    global _TOK
    from bpe_tokenizer import PureByteLevelBPETokenizer
    _TOK = PureByteLevelBPETokenizer.load(tok_dir)

def _enc(a):
    text, pre = a
    return np.asarray(_TOK.encode(clean(text, pre)), dtype=np.uint16)     # vocab 33k < 65536

def build_cache(src, name, pre, tok_dir, out_dir, nproc=None, force=False, block=512):
    """Stream a JSON array of {id,text,label}, drop exact-duplicate texts (md5), tokenise in parallel, save ragged arrays."""
    import ijson
    os.makedirs(out_dir, exist_ok=True)
    path = f"{out_dir}/{name}_{pre}.npz"
    if os.path.exists(path) and not force:
        print("cache exists:", path); return path
    nproc = nproc or max(1, os.cpu_count() or 2)
    seen, ids_l, off, ys, dids = set(), [], [0], [], []
    n_raw = n_dup = 0

    def flush(buf, pool):
        res = pool.map(_enc, [(t, pre) for t, _, _ in buf], chunksize=4)
        for (t, y, d), a in zip(buf, res):
            ids_l.append(a); off.append(off[-1] + len(a)); ys.append(y); dids.append(d)

    with Pool(nproc, initializer=_init, initargs=(tok_dir,)) as pool, open(src, "rb") as f:
        buf = []
        for rec in ijson.items(f, "item"):
            n_raw += 1
            text = rec["text"]
            h = hashlib.md5(text.encode("utf-8", "ignore")).hexdigest()
            if h in seen:
                n_dup += 1; continue
            seen.add(h)
            lab = rec.get("label")
            buf.append((text, int(lab) if lab is not None else -1, str(rec["id"])))
            if len(buf) >= block:
                flush(buf, pool); buf = []
        if buf: flush(buf, pool)
    np.savez(path, ids=np.concatenate(ids_l), off=np.asarray(off, dtype=np.int64),
             y=np.asarray(ys, dtype=np.int8), did=np.asarray(dids))
    print(f"{name}/{pre}: {n_raw} records, {n_dup} exact-duplicate texts dropped, {len(ys)} kept, {off[-1]/1e6:.1f}M tokens -> {path}")
    return path

def load_cache(path):
    z = np.load(path)
    return {k: z[k] for k in z.files}

def doc_ids(C, i):
    return C["ids"][C["off"][i]:C["off"][i + 1]]

def key_ids(tok):
    """First BPE id of each decision word (used only by pick='kw', a cheap evidence-selection heuristic)."""
    return np.array(sorted({tok.encode(" " + w)[0] for w in DECISION_WORDS}), dtype=np.int64)


# ------------------------------------------------------------------ long-document representation
def chunk_starts(n, body, overlap):
    if n <= body: return [0]
    stride = max(1, body - overlap)
    s = list(range(0, n - body + 1, stride))
    if s[-1] + body < n: s.append(n - body)          # last window always reaches the end of the document
    return s

def n_chunks(n, c):
    if c["view"] != "chunks": return 1
    return min(c["max_chunks"], len(chunk_starts(n, c["chunk_len"] - 2, c["overlap"])))

def make_chunks(ids, c, bos, eos, keys=None):
    """view: head | tail | headtail (single window)  or  chunks (sliding windows, at most max_chunks kept).
       pick (only when there are more windows than max_chunks): first | ht (head+tail) | even | kw (decision-word score)."""
    body = c["chunk_len"] - 2; n = len(ids); v = c["view"]
    if v == "head":   parts = [ids[:body]]
    elif v == "tail": parts = [ids[-body:]]
    elif v == "headtail":
        h = body // 4
        parts = [ids if n <= body else np.concatenate([ids[:h], ids[n - (body - h):]])]
    else:
        parts = [ids[s:s + body] for s in chunk_starts(n, body, c["overlap"])]
        M = c["max_chunks"]
        if len(parts) > M:
            m, pk = len(parts), c["pick"]
            if pk == "first": sel = list(range(M))
            elif pk == "ht":
                h = M // 4; sel = list(range(h)) + list(range(m - (M - h), m))
            elif pk == "even": sel = sorted(set(np.linspace(0, m - 1, M).round().astype(int).tolist()))
            elif pk == "kw":
                sc = np.array([np.isin(p, keys).mean() for p in parts])
                top = [int(i) for i in np.argsort(-sc) if i not in (0, m - 1)][:M - 2]
                sel = sorted({0, m - 1, *top})
            else: raise ValueError(pk)
            parts = [parts[i] for i in sel]
    return [np.concatenate([[bos], p.astype(np.int64), [eos]]) for p in parts]

def make_batches(C, idx, c, shuffle, rng, budget):
    """Group documents so that the total number of chunks per batch is <= budget (a doc is never split)."""
    off = C["off"]
    nch = {i: n_chunks(int(off[i + 1] - off[i]), c) for i in idx}
    order = list(idx)
    if shuffle: rng.shuffle(order)
    out = []
    for s in range(0, len(order), 256):               # bucket by chunk count inside a window -> less padding
        part = sorted(order[s:s + 256], key=lambda j: nch[j])
        cur, tot = [], 0
        for j in part:
            if cur and tot + nch[j] > budget:
                out.append(cur); cur, tot = [], 0
            cur.append(j); tot += nch[j]
        if cur: out.append(cur)
    if shuffle: rng.shuffle(out)
    return out

def subset(C, frac, seed):
    """Stratified fixed subsample (same indices for the same frac/seed -> teacher and student see the same docs)."""
    idx = np.arange(len(C["y"]))
    if frac >= 1: return idx.tolist()
    rng = np.random.RandomState(seed); out = []
    for k in np.unique(C["y"]):
        ii = idx[C["y"] == k]
        out += rng.choice(ii, max(1, int(round(frac * len(ii)))), replace=False).tolist()
    return sorted(out)

In [ ]:
%%writefile cjpe.py
import os, json, math, time, random, argparse
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from llama_scratch import Config, LlamaLM, PRESETS
from bpe_tokenizer import PureByteLevelBPETokenizer
from tqdm import tqdm
from cjpe_data import load_cache, doc_ids, make_chunks, make_batches, subset, key_ids

NC = 2
ALL_LIN = ["wq", "wk", "wv", "wo", "w_gate", "w_up", "w_down"]


# ---------------------------------------------------------------- base model (Phase 1 checkpoint)
def load_base(ckpt, keep_every=None):
    ck = torch.load(ckpt, map_location="cpu", weights_only=False)
    sd = next((ck[k] for k in ("model", "model_state_dict", "state_dict") if isinstance(ck.get(k), dict)), ck)
    sd = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()}
    cfg = ck.get("config") or ck.get("cfg")
    if isinstance(cfg, dict): cfg = Config(**cfg)
    if cfg is None:
        V, D = sd["tok_emb.weight"].shape
        L = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("layers."))
        nh = next(p["n_heads"] for p in PRESETS.values() if p["d_model"] == D)
        nkv = sd["layers.0.attn.wk.weight"].shape[0] // (D // nh)
        cfg = Config(vocab_size=V, d_model=D, n_layers=L, n_heads=nh, n_kv_heads=nkv, ffn_hidden=sd["layers.0.mlp.w_gate.weight"].shape[0])
    cfg.attn_impl = "sdpa"
    if keep_every:                                    # layer-pruned student: every k-th layer + the last
        L = cfg.n_layers; keep = sorted(set(range(0, L, keep_every)) | {L - 1})
        remap = {o: n for n, o in enumerate(keep)}; new = {}
        for k, v in sd.items():
            if k.startswith("layers."):
                p = k.split("."); o = int(p[1])
                if o not in remap: continue
                p[1] = str(remap[o]); k = ".".join(p)
            new[k] = v
        sd = new; cfg.n_layers = len(keep); print("student keeps layers", keep, flush=True)
    m = LlamaLM(cfg)
    res = m.load_state_dict(sd, strict=False)
    bad = [k for k in res.missing_keys if "lm_head" not in k]
    assert not bad and not res.unexpected_keys, (bad[:5], res.unexpected_keys[:5])
    return m


# ---------------------------------------------------------------- fine-tuning methods
def attach_peft(base, c, total_steps):
    p, r = c["peft"], c["r"]
    if p == "full": return base
    from peft import get_peft_model, LoraConfig, AdaLoraConfig
    if p == "lora":      cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, target_modules=ALL_LIN)
    elif p == "rslora":  cfg = LoraConfig(r=r, lora_alpha=2 * math.sqrt(r), lora_dropout=0.05, target_modules=ALL_LIN, use_rslora=True)
    elif p == "adalora": cfg = AdaLoraConfig(init_r=int(1.5 * r), target_r=r, lora_alpha=2 * r, lora_dropout=0.05, target_modules=ALL_LIN,
                                             tinit=int(.1 * total_steps), tfinal=int(.3 * total_steps), deltaT=10, total_step=total_steps)
    elif p in ("lorma_plus", "lorma_pi"):
        from lorma_hook import apply_lorma
        return apply_lorma(base, r=r, target_modules=ALL_LIN, variant=p.split("_")[1])
    else: raise ValueError(p)
    return get_peft_model(base, cfg)


def set_trainable_layers(model, mode, n, delta=None, emb=False):
    """Layer freezing for full fine-tuning. Freezes n transformer layers (embeddings frozen too unless emb=True).
       int: evenly spaced | bot: lowest n | top: highest n | adt_l: n smallest weight change | adt_h: n largest weight change."""
    core = model.core(); L = len(core.layers)
    if mode == "bot": fr = set(range(n))
    elif mode == "top": fr = set(range(L - n, L))
    elif mode == "int":
        fr = set(int(round(x)) for x in np.linspace(0, L - 1, n))
        for i in range(L):
            if len(fr) >= n: break
            fr.add(i)
    elif mode == "adt_l": fr = set(np.argsort(delta)[:n].tolist())
    elif mode == "adt_h": fr = set(np.argsort(delta)[-n:].tolist())
    else: raise ValueError(mode)
    for i, l in enumerate(core.layers):
        for p in l.parameters(): p.requires_grad = i not in fr
    core.tok_emb.weight.requires_grad = bool(emb)            # tied with lm_head
    return sorted(int(i) for i in fr)


# ---------------------------------------------------------------- document classifier
class CJPEModel(nn.Module):
    """Phase-1 LM encodes every chunk causally -> chunk vector (state at </s> or masked mean) -> pooling across chunks -> head -> 2 logits."""
    def __init__(self, base, c, total_steps):
        super().__init__()
        self.peft = attach_peft(base, c, total_steps)
        D = base.cfg.d_model; self.agg, self.ctype, self.gc = c["agg"], c["ctype"], c.get("grad_ckpt", True)
        self.attn_q = nn.Linear(D, 1) if self.agg in ("attn", "hier") else None
        self.pos = self.enc = None
        if self.agg == "hier":                       # chunk-level transformer: chunk vectors + chunk position -> 1 layer -> attention pooling
            self.pos = nn.Embedding(64, D)
            self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(D, 8, 2 * D, 0.1, activation="gelu", batch_first=True, norm_first=True),
                                             1, enable_nested_tensor=False)
        fd = 3 * D if self.agg == "seg" else D
        self.cls = (nn.Sequential(nn.Dropout(.1), nn.Linear(fd, D), nn.GELU(), nn.Dropout(.1), nn.Linear(D, NC)) if c["head"] == "mlp"
                    else nn.Sequential(nn.Dropout(.1), nn.Linear(fd, NC)))

    def core(self): return self.peft.get_base_model() if hasattr(self.peft, "get_base_model") else self.peft

    def head_params(self):
        ps = list(self.cls.parameters())
        for m in (self.attn_q, self.pos, self.enc):
            if m is not None: ps += list(m.parameters())
        return ps

    def hidden(self, idx):
        b = self.core(); x = b.tok_emb(idx); T = idx.size(1)
        cos = sin = None
        if b.cfg.pos_enc == "rope": cos, sin = b.rope_cos[:T], b.rope_sin[:T]
        for l in b.layers:
            if self.gc and self.training and torch.is_grad_enabled():
                x = checkpoint(lambda y, l=l: l(y, cos, sin, b.alibi_slopes, None, None, False)[0], x, use_reentrant=False)
            else:
                x, _ = l(x, cos, sin, b.alibi_slopes, None, None, False)
        return b.norm(x)

    def forward(self, ids, lens, owner, nb):
        h = self.hidden(ids); N, T, D = h.shape
        if self.ctype == "last": v = h[torch.arange(N, device=h.device), lens - 1]
        else:
            m = (torch.arange(T, device=h.device)[None] < lens[:, None]).unsqueeze(-1).to(h.dtype)
            v = (h * m).sum(1) / lens[:, None]
        v = v.float()
        counts = torch.bincount(owner, minlength=nb); C = int(counts.max())            # chunks are stored doc by doc
        start = torch.cumsum(counts, 0) - counts
        pos = torch.arange(N, device=h.device) - start[owner]
        G = v.new_zeros(nb, C, D).index_put((owner, pos), v)                           # (docs, chunks, D)
        ar = torch.arange(C, device=h.device)[None]
        M = ar < counts[:, None]; Mf = M.unsqueeze(-1).to(G.dtype)
        cnt = counts[:, None].to(G.dtype)
        if self.agg == "mean": r = (G * Mf).sum(1) / cnt
        elif self.agg == "max": r = G.masked_fill(~M.unsqueeze(-1), -1e4).max(1).values
        elif self.agg == "lastchunk": r = G[torch.arange(nb, device=h.device), counts - 1]
        elif self.agg == "seg":                                                         # head / middle / tail thirds of the judgment
            rel = (ar * 3 // counts[:, None]).clamp(max=2); dm = (G * Mf).sum(1) / cnt; parts = []
            for k in range(3):
                mk = (M & (rel == k)).unsqueeze(-1).to(G.dtype); ck = mk.sum(1)
                parts.append(torch.where(ck > 0, (G * mk).sum(1) / ck.clamp(min=1), dm))
            r = torch.cat(parts, -1)
        else:                                                                           # attn | hier
            if self.agg == "hier":
                assert C <= 64, "hier supports at most 64 chunks"
                G = self.enc(G + self.pos(ar.expand(nb, C)), src_key_padding_mask=~M)
            a = self.attn_q(G).squeeze(-1).float().masked_fill(~M, -1e4).softmax(-1)
            r = (a.unsqueeze(-1).to(G.dtype) * G).sum(1)
        return self.cls(r.float())


# ---------------------------------------------------------------- loss / metrics
def make_loss(kind, counts, dev):
    pr = torch.tensor(counts / counts.sum(), dtype=torch.float32, device=dev).clamp_min(1e-6)
    w = pr.pow(-0.5); w = w / (w * pr).sum()
    if kind == "none": return lambda z, y: F.cross_entropy(z, y)
    if kind == "cw":   return lambda z, y: F.cross_entropy(z, y, weight=w)
    if kind == "logit_adj": la = pr.log(); return lambda z, y: F.cross_entropy(z + la, y)    # inference uses raw logits -> balanced classifier
    if kind == "focal":
        def f(z, y):
            lp = F.log_softmax(z, -1).gather(1, y[:, None]).squeeze(1)
            return (-(w[y]) * (1 - lp.exp()) ** 2 * lp).sum() / w[y].sum()
        return f
    raise ValueError(kind)

def scores(Z, y, thr=0.5):
    p1 = F.softmax(Z, -1)[:, 1].numpy(); pred = (p1 > thr).astype(int)
    out = dict(acc=float(accuracy_score(y, pred)), macro_f1=float(f1_score(y, pred, average="macro", zero_division=0)))
    try: out["auc"] = float(roc_auc_score(y, p1))
    except Exception: out["auc"] = float("nan")
    return out

def tune_thr(Z, y):
    p1 = F.softmax(Z, -1)[:, 1].numpy()
    ts = np.linspace(0.2, 0.8, 61)
    return float(max(ts, key=lambda t: f1_score(y, (p1 > t).astype(int), average="macro", zero_division=0)))


# ---------------------------------------------------------------- batching + prediction
def collate(C, batch, c, pad, BOS, EOS, keys):
    chunks, owner = [], []
    for b, i in enumerate(batch):
        for p in make_chunks(doc_ids(C, i), c, BOS, EOS, keys):
            chunks.append(p); owner.append(b)
    T = max(len(p) for p in chunks)
    ids = torch.full((len(chunks), T), pad, dtype=torch.long); lens = torch.zeros(len(chunks), dtype=torch.long)
    for k, p in enumerate(chunks):
        ids[k, :len(p)] = torch.from_numpy(p); lens[k] = len(p)            # right padding is safe with causal attention
    y = torch.tensor([int(C["y"][i]) for i in batch], dtype=torch.long)
    return ids, lens, torch.tensor(owner), y

@torch.no_grad()
def predict(model, C, idx, ec, pad, BOS, EOS, keys):
    model.eval(); Z, K = [], []
    for b in tqdm(make_batches(C, idx, ec, False, None, ec.get("chunk_bs_eval", 16)), desc="eval", mininterval=30, leave=False):
        ids, lens, owner, _ = collate(C, b, ec, pad, BOS, EOS, keys)
        with torch.autocast("cuda", dtype=torch.float16):
            z = model(ids.cuda(), lens.cuda(), owner.cuda(), len(b))
        Z.append(z.float().cpu()); K += b
    Z = torch.cat(Z); pos = {k: i for i, k in enumerate(K)}
    return Z[[pos[i] for i in idx]]                                        # same order as idx


# ---------------------------------------------------------------- one experiment
def run(c):
    t0 = time.time(); dev = torch.device("cuda")
    random.seed(c["seed"]); np.random.seed(c["seed"]); torch.manual_seed(c["seed"]); rng = random.Random(c["seed"])
    c.setdefault("eval_chunks", c["max_chunks"]); c.setdefault("accum", 4); c.setdefault("evals_per_epoch", 1)
    tok = PureByteLevelBPETokenizer.load(c["tok_dir"]); pad, BOS, EOS = tok.pad_token_id, tok.bos_token_id, tok.eos_token_id
    keys = key_ids(tok) if c["pick"] == "kw" else None
    Ctr = load_cache(f"{c['cache']}/train_{c['pre']}.npz"); Cdv = load_cache(f"{c['cache']}/dev_{c['pre']}.npz")
    tr, dv = subset(Ctr, c["frac"], c["seed"]), subset(Cdv, c["dev_frac"], 0)
    ec = dict(c, max_chunks=c["eval_chunks"]); ydv = Cdv["y"][dv]
    counts = np.bincount(Ctr["y"][tr], minlength=NC).astype(np.float64)
    nbat = len(make_batches(Ctr, tr, c, True, random.Random(0), c["chunk_bs"]))
    steps = max(1, nbat // c["accum"]) * c["epochs"]                        # optimizer steps
    model = CJPEModel(load_base(c["ckpt"], c.get("keep_every")), c, steps).to(dev)
    if c["peft"] == "full":
        for p in model.core().parameters(): p.requires_grad = True
        if c.get("freeze"):
            fr = c["freeze"]
            delta = json.load(open(fr["delta"]))[fr.get("kind", "abs")] if fr["mode"].startswith("adt") else None
            c["frozen_layers"] = set_trainable_layers(model, fr["mode"], fr["n"], delta, fr.get("emb", False))
            print(f'[{c["name"]}] frozen layers: {c["frozen_layers"]}', flush=True)
    W0 = ({n: p.detach().cpu().clone() for n, p in model.core().named_parameters() if n.startswith("layers.")}
          if c.get("save_delta") else None)
    head = model.head_params(); hid = {id(p) for p in head}
    ad = [p for p in model.parameters() if p.requires_grad and id(p) not in hid]
    opt = torch.optim.AdamW([dict(params=ad, lr=c["lr"], weight_decay=0.0), dict(params=head, lr=c.get("head_lr", 1e-3), weight_decay=0.01)])
    base_lrs = [g["lr"] for g in opt.param_groups]; warm = max(1, int(.06 * steps))
    lossf = make_loss(c["bal"], counts, dev); scaler = torch.amp.GradScaler()
    TL = torch.load(c["kd_logits"]) if c.get("kd_logits") else None
    n_ev = c["evals_per_epoch"] * c["epochs"]
    eval_steps = {max(1, round(steps * k / n_ev)) for k in range(1, n_ev + 1)}
    os.makedirs(c["out"], exist_ok=True); ck_path = f"{c['out']}/{c['name']}.pt"
    best, best_state, best_s, best_step, hist = -1.0, None, None, 0, []
    step = micro = tok_count = 0; t_eval = 0.0; tt = time.time()

    for ep in range(c["epochs"]):
        model.train()
        for b in make_batches(Ctr, tr, c, True, rng, c["chunk_bs"]):
            ids, lens, owner, y = collate(Ctr, b, c, pad, BOS, EOS, keys)
            with torch.autocast("cuda", dtype=torch.float16):
                z = model(ids.to(dev), lens.to(dev), owner.to(dev), len(b))
            y = y.to(dev); loss = lossf(z.float(), y)
            if TL is not None:                       # KD: (1-a)*CE + a*T^2*KL(teacher || student)
                t = torch.stack([TL[i] for i in b]).to(dev).float(); T_, a = c["kd_T"], c["kd_alpha"]
                loss = (1 - a) * loss + a * T_ * T_ * F.kl_div(F.log_softmax(z.float() / T_, -1), F.softmax(t / T_, -1), reduction="batchmean")
            scaler.scale(loss / c["accum"]).backward(); micro += 1; tok_count += int(lens.sum())
            if micro % c["accum"]: continue
            sc = step / warm if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, steps - warm)))
            for g, l0 in zip(opt.param_groups, base_lrs): g["lr"] = l0 * sc
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
            scaler.step(opt); scaler.update(); step += 1
            if c["peft"] == "adalora": model.peft.base_model.update_and_allocate(step)
            opt.zero_grad(set_to_none=True)
            if step % 10 == 0: hist.append((step, float(loss.detach())))
            if step % 25 == 0 or step == steps:
                el = time.time() - tt
                print(f'[{c["name"]}] step {step}/{steps} ep {ep} loss {loss.item():.4f} elapsed {el/60:.1f}m eta {el/step*(steps-step)/60:.1f}m', flush=True)
            if step in eval_steps:
                te = time.time(); Z = predict(model, Cdv, dv, ec, pad, BOS, EOS, keys); s = scores(Z, ydv)
                print(f'[{c["name"]}] EVAL step {step}: {s}', flush=True); t_eval += time.time() - te
                if s[c.get("sel", "macro_f1")] > best:
                    best, best_s, best_step = s[c.get("sel", "macro_f1")], s, step
                    best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
                    torch.save(dict(cfg=c, state=best_state, thr=0.5, step=step), ck_path)         # saved at every improvement
                model.train()
            if step >= steps: break
    train_min = (time.time() - tt - t_eval) / 60
    model.load_state_dict(best_state, strict=False)
    Zd = predict(model, Cdv, dv, ec, pad, BOS, EOS, keys); yd = ydv
    thr = tune_thr(Zd, yd); s05, sth = scores(Zd, yd), scores(Zd, yd, thr)
    torch.save(dict(cfg=c, state=best_state, thr=thr, step=best_step), ck_path)
    json.dump(hist, open(f"{c['out']}/{c['name']}_loss.json", "w"))
    if W0 is not None:                               # per-layer weight change (for ADT-L / ADT-H)
        a, r = {}, {}
        for n, p in model.core().named_parameters():
            if n in W0:
                l = int(n.split(".")[1]); d = (p.detach().cpu().float() - W0[n].float()).pow(2).sum().item()
                a[l] = a.get(l, 0.0) + d; r[l] = r.get(l, 0.0) + W0[n].float().pow(2).sum().item()
        L = max(a) + 1
        json.dump(dict(abs=[math.sqrt(a[l]) for l in range(L)], rel=[math.sqrt(a[l] / r[l]) for l in range(L)]), open(f"{c['out']}/{c['name']}_delta.json", "w"))
    tp = sum(p.numel() for p in ad)
    return dict(name=c["name"], cfg=c, trainable_adapter_params=tp, trainable_pct=100 * tp / sum(p.numel() for p in model.core().parameters()),
                peak_mem_gb=torch.cuda.max_memory_allocated() / 1e9, minutes=(time.time() - t0) / 60, train_minutes=train_min,
                tokens_per_sec=tok_count / max(1.0, train_min * 60), best_step=best_step, steps=steps,
                acc=s05["acc"], macro_f1=s05["macro_f1"], auc=s05["auc"], thr=thr, macro_f1_thr=sth["macro_f1"], acc_thr=sth["acc"])


# ---------------------------------------------------------------- reload / inference helpers (used after training)
def load_trained(name, out):
    ck = torch.load(f"{out}/{name}.pt", map_location="cpu", weights_only=False); c = ck["cfg"]
    model = CJPEModel(load_base(c["ckpt"], c.get("keep_every")), c, 100).cuda()
    res = model.load_state_dict(ck["state"], strict=False); assert not res.unexpected_keys, res.unexpected_keys[:5]
    return model, c, ck.get("thr", 0.5)

def logits_for(name, out, split, frac=1.0, seed=0):
    """Logits of a saved model on a split ('train' | 'dev' | 'test') -> (Z, y, doc indices)."""
    model, c, thr = load_trained(name, out); tok = PureByteLevelBPETokenizer.load(c["tok_dir"])
    pad, BOS, EOS = tok.pad_token_id, tok.bos_token_id, tok.eos_token_id; keys = key_ids(tok) if c["pick"] == "kw" else None
    C = load_cache(f"{c['cache']}/{split}_{c['pre']}.npz"); idx = subset(C, frac, seed)
    ec = dict(c, max_chunks=c.get("eval_chunks", c["max_chunks"]))
    Z = predict(model, C, idx, ec, pad, BOS, EOS, keys); del model; torch.cuda.empty_cache()
    return Z, C["y"][idx], idx, C["did"][idx], thr

def dump_teacher_logits(name, out, frac, seed):
    """Teacher logits on exactly the training subset a student with the same (frac, seed) will use -> {doc index: logits}."""
    Z, _, idx, _, _ = logits_for(name, out, "train", frac, seed)
    path = f"{out}/{name}_trainlogits_f{frac}_s{seed}.pt"; torch.save({int(i): z for i, z in zip(idx, Z)}, path); return path


if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--jobs"); a = ap.parse_args()
    for c in json.load(open(a.jobs)):
        res = f"{c['out']}/results.jsonl"
        if os.path.exists(res) and any(json.loads(l)["name"] == c["name"] for l in open(res)): continue
        try: r = run(c)
        except Exception:
            import traceback, gc; print(f"[{c['name']}] FAILED", flush=True); traceback.print_exc(); gc.collect(); torch.cuda.empty_cache(); continue
        with open(res, "a") as f: f.write(json.dumps(r) + "\n")
        torch.cuda.empty_cache()

## Tokenise once and cache (preprocessing `norm`)  – every later experiment only re-chunks cached ids

In [ ]:
import sys, time, os, json, numpy as np
sys.path.insert(0, ".")
CKPT_PATH = "/kaggle/input/datasets/koooshiieee/phase2v1/final_model.pt"   # Phase 1 model
TOK_DIR   = "/kaggle/working/tok_phase1"
WORK      = "/kaggle/working"
CACHE     = f"{WORK}/tok_cache"
OUT       = f"{WORK}/cjpe"
os.makedirs(OUT, exist_ok=True)
from cjpe_data import build_cache
SRC = {"train": P["train.json"], "dev": P["dev.json"], "test": TEST[0]}   # dev is NEVER trained on

def ensure_cache(pre):                       # call with "raw" / "mask" before the preprocessing ablation
    t0 = time.time()
    for split, path in SRC.items(): build_cache(path, split, pre, TOK_DIR, CACHE)
    print(f"{pre}: done in {(time.time()-t0)/60:.1f} min")

ensure_cache("norm")

In [ ]:
# Length statistics in TOKENS (decides chunk length / number of chunks) + a sanity check of the chunker
from cjpe_data import load_cache, doc_ids, make_chunks, n_chunks
from bpe_tokenizer import PureByteLevelBPETokenizer
tok = PureByteLevelBPETokenizer.load(TOK_DIR)
for s in ("train", "dev", "test"):
    n = np.diff(np.load(f"{CACHE}/{s}_norm.npz")["off"])
    print(f"{s:5s} docs={len(n):6d} | tokens p50/p90/p99 = {[int(np.percentile(n, q)) for q in (50, 90, 99)]} | "
          + " ".join(f">{k}: {100*(n>k).mean():.0f}%" for k in (512, 1024, 2048, 4096, 8192)))
n = np.diff(np.load(f"{CACHE}/train_norm.npz")["off"])
for L, ov, M in [(1024, 128, 4), (1024, 128, 8), (1024, 128, 16), (512, 64, 16)]:
    c = dict(view="chunks", chunk_len=L, overlap=ov, max_chunks=M, pick="ht")
    k = np.array([n_chunks(int(x), c) for x in n[:5000]])
    print(f"chunk_len={L} overlap={ov} max_chunks={M}: mean chunks/doc {k.mean():.2f}, docs hitting the cap {100*(k==M).mean():.0f}%")
C = load_cache(f"{CACHE}/dev_norm.npz")
ch = make_chunks(doc_ids(C, 0), dict(view="chunks", chunk_len=1024, overlap=128, max_chunks=8, pick="ht"), tok.bos_token_id, tok.eos_token_id)
print("dev doc 0 ->", len(ch), "chunks, lengths", [len(x) for x in ch], "| label", int(C["y"][0]))
print(tok.decode(ch[0][1:60].tolist()), "...")
print("... ", tok.decode(ch[-1][-60:-1].tolist()))

In [ ]:
# Where is the signal? CPU-only bag-of-token-ids + logistic regression on 6,000 train docs, evaluated on dev.
# If "last" >> "first" use tail-heavy chunk selection; if "full" wins use all-chunk pooling.
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
Ctr, Cdv = load_cache(f"{CACHE}/train_norm.npz"), load_cache(f"{CACHE}/dev_norm.npz")
def bow(Cx, idx, region, V=33000):
    rows, cols, vals = [], [], []
    for r_, i in enumerate(idx):
        d = doc_ids(Cx, i); m = len(d)
        seg = {"first": d[:1024], "mid": d[max(0, m//2-512): m//2+512], "last": d[-1024:], "full": d}[region]
        u, ct = np.unique(seg, return_counts=True); rows += [r_]*len(u); cols += u.tolist(); vals += ct.tolist()
    return csr_matrix((vals, (rows, cols)), shape=(len(idx), V), dtype=np.float32)
rs = np.random.RandomState(0); tri = rs.choice(len(Ctr["y"]), 6000, replace=False); dvi = np.arange(len(Cdv["y"]))
for region in ("first", "mid", "last", "full"):
    tf = TfidfTransformer(sublinear_tf=True).fit(bow(Ctr, tri, region))
    lr = LogisticRegression(max_iter=300, C=4).fit(tf.transform(bow(Ctr, tri, region)), Ctr["y"][tri])
    p = lr.predict(tf.transform(bow(Cdv, dvi, region)))
    print(f"{region:5s}  dev acc {accuracy_score(Cdv['y'], p):.3f}  macro-F1 {f1_score(Cdv['y'], p, average='macro'):.3f}")
ln = np.diff(Cdv["off"]); print("length-only baseline corr with label:", np.corrcoef(np.log(ln), Cdv["y"])[0, 1].round(3))

## Job launcher (2 x T4: jobs are split across both GPUs, finished jobs are skipped on re-run)

In [ ]:
import json, copy, subprocess, time
BASE = dict(ckpt=CKPT_PATH, tok_dir=TOK_DIR, cache=CACHE, out=OUT,
            peft="lora", lr=3e-4, r=16, pre="norm",
            view="chunks", chunk_len=1024, overlap=128, max_chunks=4, pick="ht",   # long-document representation
            ctype="last", agg="mean", head="linear", bal="logit_adj",                    # chunk vector / pooling / head / imbalance
            chunk_bs=8, accum=4, epochs=1, frac=0.15, dev_frac=0.5, evals_per_epoch=1,
            sel="macro_f1", seed=67, grad_ckpt=True)
REP_KEYS = ("pre", "view", "chunk_len", "overlap", "max_chunks", "pick", "ctype", "agg", "head", "bal", "chunk_bs")

def J(name, base, **kw):
    d = copy.deepcopy(base); d.update(kw); d["name"] = name; return d

def results(out=None):
    p = f"{out or OUT}/results.jsonl"; return [json.loads(l) for l in open(p)] if os.path.exists(p) else []

def run_jobs(jobs, out=None):
    out = out or jobs[0]["out"]
    done = {r["name"] for r in results(out)}; jobs = [j for j in jobs if j["name"] not in done]
    if not jobs: print("nothing to run"); return
    names = {j["name"] for j in jobs}; procs = []; t0 = time.time()
    for g in range(2):
        mine = jobs[g::2]
        if not mine: continue
        jf = f"{out}/jobs_{g}.json"; json.dump(mine, open(jf, "w"))
        procs.append(subprocess.Popen(["python", "-u", "cjpe.py", "--jobs", jf], env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g)),
                                      stdout=open(f"{out}/log_{g}.txt", "a"), stderr=subprocess.STDOUT))
    while any(p.poll() is None for p in procs):
        time.sleep(60)
        n = len([r for r in results(out) if r["name"] in names])
        print(f"{time.strftime('%H:%M:%S')} | {n}/{len(jobs)} jobs done | {(time.time()-t0)/60:.0f} min elapsed")
        for g in range(len(procs)):
            try: print(f"   GPU{g}:", open(f"{out}/log_{g}.txt").read().strip().splitlines()[-1])
            except Exception: pass
    missing = names - {r["name"] for r in results(out)}
    if missing: print("FAILED / MISSING (see log_*.txt):", sorted(missing))

def pick(pred, key="macro_f1", out=None):
    return max([r for r in results(out) if pred(r["name"])], key=lambda r: r[key])

def stage(tag, prev, variants):               # greedy stage: keep the best of {previous best} U {variants}
    names = {prev["name"]}
    run_jobs([J(f"{tag}_{n}", prev["cfg"], **kw) for n, kw in variants.items()])
    return pick(lambda n: n in names or n.startswith(tag + "_"))

def est_hours(cfg, tps):                      # wall-clock estimate for ONE run on ONE GPU, from a measured tokens/sec
    n = np.diff(np.load(f"{cfg['cache']}/train_{cfg['pre']}.npz")["off"])
    cap = cfg["max_chunks"] * (cfg["chunk_len"] - 2) if cfg["view"] == "chunks" else cfg["chunk_len"]
    return np.minimum(n, cap).sum() * cfg["frac"] * cfg["epochs"] / tps / 3600

import pandas as pd
def table(prefixes, cols=("name", "macro_f1", "acc", "auc", "macro_f1_thr", "thr", "trainable_pct", "train_minutes", "tokens_per_sec", "peak_mem_gb")):
    rows = [{k: r.get(k) for k in cols} for r in results() if r["name"].startswith(prefixes)]
    return pd.DataFrame(rows).sort_values("macro_f1", ascending=False).round(4)

## Smoke test (run ONCE, ~15-25 min; checks every code path on 1% of the data, then delete it)

In [ ]:
'''SM_OUT = f"{WORK}/cjpe_smoke"; os.makedirs(SM_OUT, exist_ok=True)
SM = dict(BASE, out=SM_OUT, frac=0.003, dev_frac=0.02, epochs=1, max_chunks=2, evals_per_epoch=1)
run_jobs([J("s_lorma", SM, peft="lorma_plus"), J("s_ada", SM, peft="adalora"),
          J("s_full", SM, peft="full", lr=3e-5, save_delta=True)])
run_jobs([J("s_top", SM, peft="full", lr=3e-5, freeze=dict(mode="top", n=7))])
import importlib, cjpe; importlib.reload(cjpe)
tp = cjpe.dump_teacher_logits("s_ada", SM_OUT, 0.003, 67)
run_jobs([J("s_kd", SM, peft="full", lr=5e-5, keep_every=7, kd_logits=tp, kd_T=2.0, kd_alpha=0.5)])
for r in results(SM_OUT): print(r["name"], round(r["macro_f1"], 3), f'{r["tokens_per_sec"]:.0f} tok/s')'''

In [ ]:
'''# after the smoke test passes:
import shutil; shutil.rmtree(SM_OUT, ignore_errors=True)'''

## Stage R – representation ablation (LoRA probe, 15% of train, 1 epoch, 50% of dev)
Greedy and cumulative like Phase 3: each stage keeps the winner. Differences below ~2 points of dev macro-F1 are within noise, so prefer the cheaper option on ties.

In [ ]:
# R0 + R1: window / chunk length / overlap / number of chunks   (single-window baselines vs sliding-window chunking)
run_jobs([J("r0_base", BASE)])
b0 = pick(lambda n: n == "r0_base"); print("tokens/sec (LoRA):", round(b0["tokens_per_sec"]), "| dev macro-F1:", round(b0["macro_f1"], 4))
b1 = stage("r1", b0, {
    "head1024": dict(view="head", max_chunks=1), "tail1024": dict(view="tail", max_chunks=1), "headtail1024": dict(view="headtail", max_chunks=1)})
print(b1["name"], b1["macro_f1"]); table(("r0_", "r1_"))

In [ ]:
'''# R2: which chunks to keep when the document has more windows than max_chunks (selective context)
b2 = stage("r2", b1, {"first": dict(pick="first"), "even": dict(pick="even"), "kw": dict(pick="kw")})
print(b2["name"], b2["macro_f1"]); table(("r2_",))'''

In [ ]:
'''# R3: pooling across chunks (mean is the base) – max, learned attention, head/middle/tail segments, chunk-level transformer (hierarchical), last chunk only
b3 = stage("r3", b2, {"max": dict(agg="max"), "attn": dict(agg="attn"), "seg": dict(agg="seg"), "hier": dict(agg="hier"), "lastchunk": dict(agg="lastchunk")})
print(b3["name"], b3["macro_f1"]); table(("r3_",))'''

In [ ]:
'''# R4: chunk vector (state at </s> vs masked mean) and classification head (linear vs MLP)
b4 = stage("r4", b3, {"cmean": dict(ctype="mean"), "mlp": dict(head="mlp"), "cmean_mlp": dict(ctype="mean", head="mlp")})
print(b4["name"], b4["macro_f1"]); table(("r4_",))'''

In [ ]:
'''# R5: preprocessing (needs two more tokenisation passes, ~10-20 min each; train is tokenised in parallel on all CPU cores)
ensure_cache("raw"); ensure_cache("mask")
b5 = stage("r5", b4, {"raw": dict(pre="raw"), "mask": dict(pre="mask")})
print(b5["name"], b5["macro_f1"]); table(("r5_",))'''

In [ ]:
# R6: class imbalance (train is 59/41, dev and test are 50/50). Threshold tuning on dev is reported for every run (macro_f1_thr).
# b6 = stage("r6", b5, {"cw": dict(bal="cw"), "logit_adj": dict(bal="logit_adj"), "focal": dict(bal="focal")})

# REP_BEST = {k: b6["cfg"][k] for k in REP_KEYS}; print("REP_BEST =", REP_BEST); table(("r0_", "r1_", "r2_", "r3_", "r4_", "r5_", "r6_"))

REP_BEST = dict(pre="norm", view="chunks", chunk_len=1024, overlap=128, max_chunks=4, pick="ht",
                ctype="last", agg="mean", head="linear", bal="logit_adj", chunk_bs=8)
PBASE = dict(BASE, **REP_BEST, accum=2, frac=0.1, epochs=1, dev_frac=0.3, evals_per_epoch=2)

## Stage P – fine-tuning methods x learning-rate grid on the winning representation
10% of train, 2 epochs, full dev. Pick the best LR per method by dev macro-F1, then a small rank sweep for the top 3 PEFT methods.

In [ ]:
GRID = {"lora": 5e-4, "lorma_plus": 1e-3, "adalora": 1e-3, "full": 5e-5}
# run_jobs([J(f"g_{m}_{lr}", PBASE, peft=m, lr=lr) for m, lr in GRID.items()])
run_jobs([J(f"g_{m}_{lr}", PBASE, peft=m, lr=lr, save_delta=(m == "full")) for m, lr in GRID.items()])
def best_per_method(prefixes=("g_", "gr_")):
    by = {}
    for r in results():
        if r["name"].startswith(prefixes):
            m = r["cfg"]["peft"]
            if m not in by or r["macro_f1"] > by[m]["macro_f1"]: by[m] = r
    return by
by_arch = best_per_method(); table(("g_",))

In [ ]:
'''# rank sweep (r = 8 / 16 / 32) for the three best PEFT methods at their best LR
top = [b for b in sorted(by_arch.values(), key=lambda r: -r["macro_f1"]) if b["cfg"]["peft"] != "full"][:3]
run_jobs([J(f"gr_{b['cfg']['peft']}_r{r}", b["cfg"], r=r) for b in top for r in (8, 32) if r != b["cfg"]["r"]])
by_arch = best_per_method(); print({m: (r["name"], round(r["macro_f1"], 4)) for m, r in by_arch.items()}); table(("g_", "gr_"))'''

## Stage F – full-data training of every method (best LR/rank) – saves the best-dev checkpoint of each
Dev is evaluated 4x per epoch and the checkpoint is overwritten whenever dev macro-F1 improves. Check the time estimate first; if it is too long lower `FINAL_FRAC` or `FINAL_EPOCHS` (and say so in the report).

In [ ]:
'''FINAL_FRAC, FINAL_EPOCHS = 1.0, 2
FINAL = dict(frac=0.05, epochs=1, dev_frac=0.3, evals_per_epoch=2)
est = {m: est_hours(dict(r["cfg"], **FINAL), r["tokens_per_sec"]) for m, r in by_arch.items()}
print({m: round(h, 1) for m, h in est.items()}, "GPU-hours each | total", round(sum(est.values()), 1), "| ~", round(sum(est.values()) / 2, 1), "h wall-clock on 2 GPUs")'''

In [ ]:
'''run_jobs([J(f"f_{m}", r["cfg"], **FINAL) for m, r in by_arch.items()])
table(("f_",))'''

## Stage Z – layer freezing / unfreezing (full fine-tuning, N layers frozen, embeddings frozen too)
INT: evenly spaced | BOT: lowest N | TOP: highest N | ADT-L / ADT-H: N layers with the smallest / largest weight change measured in a short probe run of full fine-tuning.

In [ ]:
ZB = dict(by_arch["full"]["cfg"], save_delta=False)
DELTA = f"{OUT}/g_full_5e-05_delta.json"
dl = json.load(open(DELTA)); print("per-layer |dW|:", [round(x, 3) for x in dl["abs"]])
N_FROZEN = 7                                                     # of 14 layers; also try 4 and 10 if time allows
run_jobs([J(f"z_{m}_n{N_FROZEN}", ZB, freeze=dict(mode=m, n=N_FROZEN, delta=DELTA)) for m in ("int", "bot", "top", "adt_l")])
table(("g_full", "z_"))


## Stage KD – knowledge distillation
Teacher = best full-data model. Student = Phase-1 weights with layers pruned to [0, 7, 13] (3 layers), fully fine-tuned.
Compare: student with CE only vs student with 0.5·CE + 0.5·T²·KL (T=2). The teacher scores exactly the training subset the student uses (same frac + seed).

In [ ]:
import importlib, cjpe; importlib.reload(cjpe)
teacher = pick(lambda n: n.startswith("g_")); print("teacher:", teacher["name"], round(teacher["macro_f1"], 4))
KD_FRAC = 0.1
tp = cjpe.dump_teacher_logits(teacher["name"], OUT, KD_FRAC, teacher["cfg"]["seed"])
STUDENT = dict(teacher["cfg"], peft="full", lr=5e-5, keep_every=7, freeze=None, frac=KD_FRAC, epochs=1, dev_frac=1.0, evals_per_epoch=2,save_delta=False)
run_jobs([J("kd_student_ce", STUDENT), J("kd_student_kd", STUDENT, kd_logits=tp, kd_T=2.0, kd_alpha=0.5)])
table(("f_", "kd_"))

## Evaluation – dev (selection split) and test (reported once, never used for selection)

In [ ]:
import importlib, cjpe; importlib.reload(cjpe)
import torch, torch.nn.functional as F
LOGDIR = f"{OUT}/logits"; os.makedirs(LOGDIR, exist_ok=True)

def get_logits(name, split):
    p = f"{LOGDIR}/{name}_{split}.pt"
    if not os.path.exists(p):
        Z, y, idx, did, thr = cjpe.logits_for(name, OUT, split); torch.save(dict(Z=Z, y=y, did=list(did), thr=thr), p)
    return torch.load(p, weights_only=False)

def evaluate(names):
    rows = []
    for n in names:
        d, t = get_logits(n, "dev"), get_logits(n, "test"); thr = d["thr"]
        sd, st, st05 = cjpe.scores(d["Z"], d["y"], thr), cjpe.scores(t["Z"], t["y"], thr), cjpe.scores(t["Z"], t["y"], 0.5)
        rows.append(dict(model=n, thr=round(thr, 3), dev_acc=sd["acc"], dev_f1=sd["macro_f1"], test_acc=st["acc"], test_f1=st["macro_f1"],
                         test_f1_thr0_5=st05["macro_f1"], test_auc=st["auc"]))
    return pd.DataFrame(rows).round(4)

final_names = sorted(r["name"] for r in results() if r["name"].startswith(("g_", "z_", "kd_")))
EVAL = evaluate(final_names); EVAL.to_csv(f"{OUT}/eval_dev_test.csv", index=False); EVAL

In [ ]:
# Submission. >>> The output format below ({id: 0/1} inside prediction.json inside submission.zip, as in Phase 3) is an ASSUMPTION:
# compare with the task page and change write_submission() if the required format differs. <<<
import zipfile
def ensemble(names):
    Pd = sum(F.softmax(get_logits(n, "dev")["Z"], -1) for n in names) / len(names)
    Pt = sum(F.softmax(get_logits(n, "test")["Z"], -1) for n in names) / len(names)
    yd, yt = get_logits(names[0], "dev")["y"], get_logits(names[0], "test")["y"]
    thr = cjpe.tune_thr(torch.log(Pd), yd)
    return Pd, Pt, yd, yt, thr, get_logits(names[0], "test")["did"]

def write_submission(names, pred_path=f"{OUT}/prediction.json", zip_path=f"{OUT}/submission.zip"):
    Pd, Pt, yd, yt, thr, did = ensemble(names)
    pred = {str(i): int(p > thr) for i, p in zip(did, Pt[:, 1].tolist())}
    assert len(pred) == len(did) and set(pred.values()) <= {0, 1}
    json.dump(pred, open(pred_path, "w"))
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z: z.write(pred_path, "prediction.json")
    print("dev (ensemble):", cjpe.scores(torch.log(Pd), yd, thr), "| test:", cjpe.scores(torch.log(Pt), yt, thr), "| thr", round(thr, 3), "|", len(pred), "predictions")

def validate_submission(zip_path=f"{OUT}/submission.zip"):
    with zipfile.ZipFile(zip_path) as z: assert z.namelist() == ["prediction.json"]; pred = json.loads(z.read("prediction.json"))
    C = load_cache(f"{CACHE}/test_norm.npz"); assert set(pred) == set(map(str, C["did"])) and set(pred.values()) <= {0, 1}
    print("submission valid:", len(pred), "docs")

best_name = EVAL.sort_values("dev_f1", ascending=False).iloc[0]["model"]; print("best on DEV:", best_name)
write_submission([best_name]); validate_submission()
# optional ensemble of the top-3 dev models: write_submission(list(EVAL.sort_values("dev_f1", ascending=False).model[:3]))

In [ ]:
for n in final_names:
    write_submission([n], pred_path=f"{OUT}/pred_{n}.json", zip_path=f"{OUT}/submission_{n}.zip")

## Results table, loss curves, upload

In [ ]:
import matplotlib.pyplot as plt
ALLR = table(("r", "g_", "f_", "z_", "kd_", "probe"), cols=("name", "macro_f1", "acc", "auc", "macro_f1_thr", "trainable_pct", "train_minutes", "peak_mem_gb"))
ALLR.to_csv(f"{OUT}/all_results.csv", index=False)
def plot(names, path, title):
    plt.figure(figsize=(8, 5))
    for n in names:
        h = json.load(open(f"{OUT}/{n}_loss.json")); plt.plot([a for a, _ in h], [b for _, b in h], label=n)
    plt.xlabel("optimizer step"); plt.ylabel("train loss"); plt.title(title); plt.legend(fontsize=6); plt.savefig(path, dpi=150); plt.close()
# plot([r["name"] for r in results() if r["name"].startswith("g_")], f"{OUT}/loss_lr_grid.png", "method x LR grid")
plot([r["name"] for r in results() if r["name"].startswith("g_")], f"{OUT}/loss_final.png", "full-data runs")
plot([r["name"] for r in results() if r["name"].startswith(("z_", "kd_"))], f"{OUT}/loss_freeze_kd.png", "layer freezing and KD")
ALLR

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN); REPO = f"{api.whoami()['name']}/legal-cjpe-phase4-v1"
api.create_repo(REPO, repo_type="model", private=True, exist_ok=True)
up = lambda p, dst: api.upload_file(path_or_fileobj=p, path_in_repo=dst, repo_id=REPO, repo_type="model")
'''for r in results():                                                   # every trained model (trainable weights of the best-dev checkpoint)
    for suf in (".pt", "_loss.json", "_delta.json"):
        f = f"{OUT}/{r['name']}{suf}"
        if os.path.exists(f): up(f, f"models/{r['name']}{suf}")'''
for f in ("results.jsonl", "eval_dev_test.csv", "all_results.csv", "loss_lr_grid.png", "loss_final.png", "loss_freeze_kd.png", "prediction.json", "submission.zip"):
    if os.path.exists(f"{OUT}/{f}"): up(f"{OUT}/{f}", f)
for f in ("cjpe.py", "cjpe_data.py", "llama_scratch.py", "bpe_tokenizer.py", "lorma_hook.py"): up(f, f"code/{f}")
for f in ("vocab.json", "merges.json", "tokenizer_config.json"): up(f"{TOK_DIR}/{f}", f"tokenizer/{f}")
print("Uploaded to https://huggingface.co/" + REPO)